# Cell 1 — Project Introduction

## SkinCancerTracker: Google Colab Environment Setup

This notebook provides an automated, reproducible setup pipeline for the **SkinCancerTracker** research project in Google Colab. It mounts Google Drive, clones or accesses the repository, audits the CUDA/GPU acceleration environment, installs all pinned dependencies from `requirements.txt`, initializes Git submodules (`external/panderm` and `external/dermfm-zero`), and executes diagnostic smoke tests on package imports and Weights & Biases tracking.

In [ ]:
# Cell 2 — Mount Google Drive
from pathlib import Path
import os

try:
    from google.colab import drive
    drive.mount("/content/drive")
    print("Google Drive successfully mounted.")
except ImportError:
    print("Running outside Google Colab; skipping Drive mount.")

# Configurable base path for persistent Drive storage
DRIVE_PROJECT_BASE = Path("/content/drive/MyDrive/SkinCancerTracker")
print(f"Configured Drive target directory: {DRIVE_PROJECT_BASE}")

In [ ]:
# Cell 3 — Clone or Access the Project Repository
import subprocess
import os
from pathlib import Path

# Configuration (update repo URL if needed)
REPO_URL = "https://github.com/placeholder-org/SkinCancerTracker.git"
LOCAL_PROJECT_DIR = Path("/content/SkinCancerTracker")

if LOCAL_PROJECT_DIR.exists() and (LOCAL_PROJECT_DIR / ".git").exists():
    print(f"Project repository already exists at: {LOCAL_PROJECT_DIR}")
    os.chdir(LOCAL_PROJECT_DIR)
    subprocess.run(["git", "status"], check=True)
else:
    print(f"Directory does not exist or is current root. Working from: {os.getcwd()}")
    # If already running inside repo clone
    if Path("pyproject.toml").exists():
        LOCAL_PROJECT_DIR = Path(os.getcwd())
        print(f"Active workspace detected at: {LOCAL_PROJECT_DIR}")
    else:
        print(f"Cloning {REPO_URL} into {LOCAL_PROJECT_DIR}...")
        subprocess.run(["git", "clone", REPO_URL, str(LOCAL_PROJECT_DIR)], check=True)
        os.chdir(LOCAL_PROJECT_DIR)

print(f"Active working directory: {os.getcwd()}")

In [ ]:
# Cell 4 — Verify Python and Hardware
import sys
import torch

print(f"Python Version: {sys.version.split()[0]}")
print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")

if torch.cuda.is_available():
    device_id = torch.cuda.current_device()
    gpu_name = torch.cuda.get_device_name(device_id)
    total_mem = torch.cuda.get_device_properties(device_id).total_memory / (1024 ** 3)
    print(f"Active GPU ({device_id}): {gpu_name}")
    print(f"Total GPU Memory: {total_mem:.2f} GB")
else:
    print("WARNING: No GPU detected! To enable GPU in Colab, go to: Runtime -> Change runtime type -> Hardware accelerator -> T4 or A100 GPU.")

In [ ]:
# Cell 5 — Install Dependencies
# Note: While environment.yml is provided for local Conda environments,
# Google Colab standardly utilizes pip. We install directly from requirements.txt.
import subprocess
import sys

print("Installing dependencies from requirements.txt...")
res = subprocess.run([sys.executable, "-m", "pip", "install", "-r", "requirements.txt"], check=True)
print("Dependencies installed successfully.")

In [ ]:
# Cell 6 — Install the Project Package in Editable Mode
import subprocess
import sys

print("Installing skincancertracker in editable mode (-e .)...")
subprocess.run([sys.executable, "-m", "pip", "install", "-e", "."], check=True)
print("Package successfully installed.")

In [ ]:
# Cell 7 — Configure Git Submodules
import subprocess
from pathlib import Path

print("Initializing and updating Git submodules (PanDerm & DermFM-Zero)...")
cmd = ["git", "submodule", "update", "--init", "--recursive"]
subprocess.run(cmd, check=True)

panderm_exists = (Path("external/panderm") / "README.md").exists()
dermfm_exists = (Path("external/dermfm-zero") / "README.md").exists()
print(f"PanDerm Submodule status: {'FOUND' if panderm_exists else 'MISSING'}")
print(f"DermFM-Zero Submodule status: {'FOUND' if dermfm_exists else 'MISSING'}")
assert panderm_exists, "PanDerm submodule initialization failed."
assert dermfm_exists, "DermFM-Zero submodule initialization failed."

In [ ]:
# Cell 8 — Verify Project Imports
print("Testing modular package imports from src...")
import src
from src.data import SkinLesionDataset, get_train_transforms, get_eval_transforms
from src.models import build_model
from src.losses import build_loss
from src.eval import WandbLogger
from src.utils import load_config, set_seed

print(f"SkinCancerTracker package v{src.__version__} imported successfully!")

# Test baseline config loading
config = load_config("configs/baseline.yaml")
print(f"Loaded baseline config for experiment: {config.experiment_name}")

In [ ]:
# Cell 9 — Verify W&B Availability
try:
    import wandb
    print(f"Weights & Biases (wandb) v{wandb.__version__} is installed and available.")
    # Test offline logger instantiation
    logger = WandbLogger(config={"wandb": {"enabled": False}})
    print("WandbLogger offline test initialization passed.")
except ImportError as e:
    print(f"ERROR: wandb is not installed ({e}). Check requirements.txt.")
    raise

In [ ]:
# Cell 10 — Final Environment Report
import os
import sys
import torch
from pathlib import Path

gpu_desc = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "None (CPU Only)"
submodules_ready = (Path("external/panderm").exists() and Path("external/dermfm-zero").exists())

report = f"""
================================================================================
SKINCANCERTRACKER — GOOGLE COLAB ENVIRONMENT SUMMARY
================================================================================
Project Path:             {os.getcwd()}
Python Version:           {sys.version.split()[0]}
PyTorch Version:          {torch.__version__}
CUDA Available:           {torch.cuda.is_available()}
Hardware / GPU:           {gpu_desc}
Submodules Status:        {'INITIALIZED' if submodules_ready else 'INCOMPLETE'}
Package Installation:     SUCCESSFUL (editable mode)
Config & W&B Validation:  SUCCESSFUL (offline verified)
================================================================================
"""
print(report)